# Preprocesamiento e Ingeniería de Variables

In [157]:
import numpy as np
from ucimlrepo import fetch_ucirepo
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.exceptions import NotFittedError
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set_theme(style="whitegrid")

## Carga y limpieza de datos

In [134]:
education_mapping = {
    "Preschool": 1,
    "1st-4th": 2,
    "5th-6th": 3,
    "7th-8th": 4,
    "9th": 5,
    "10th": 6,
    "11th": 7,
    "12th": 8,
    "HS-grad": 9,
    "Some-college": 10,
    "Assoc-voc": 11,
    "Assoc-acdm": 12,
    "Bachelors": 13,
    "Masters": 14,
    "Prof-school": 15,
    "Doctorate": 16
}

In [135]:
adult = fetch_ucirepo(id=2)
df_raw = adult.data.original

df_raw.replace('?', np.nan, inplace=True)
df_raw['income'] = df_raw['income'].str.strip().replace({'<=50K.': '<=50K', '>50K.': '>50K'})
df_raw['native-country'] = df_raw['native-country'].replace({
    'South': 'South-Korea',
    'Hong': 'Hong-Kong'
})

df_raw.drop(columns = ['fnlwgt'], inplace=True)
df_raw

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K
48838,64,NaN,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K
48839,38,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K
48840,44,Private,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K


## Ingeniería de variables

### Agrupación de nacionalidades

La variable `native-country` tiene demasiadas categorías, esto puede resultar en un problema de alta dimensionalidad para el modelo al codificarlas. Además, al registrar nuevas nacionalidades, el modelo no las reconocerá; por lo cual, vamos a regionalizar las nacionalidades para evitar esta fuga de información.

In [136]:
df_trat = df_raw.copy()

df_trat['income_num'] = df_trat['income'].map({'<=50K': 0, '>50K': 1})

In [137]:
def regionalizar_paises(X):
    # 1. Definimos el diccionario de mapeo
    X_copy = X.copy()
    region_mapping = {
    # Norteamérica
    'United-States': 'EEUU',
    'Canada': 'Norteamerica_Otros',
    'Outlying-US(Guam-USVI-etc)': 'Norteamerica_Otros',
    
    # Latinoamérica y Caribe
    'Mexico': 'Latinoamerica', 'Puerto-Rico': 'Latinoamerica', 'El-Salvador': 'Latinoamerica', 
    'Cuba': 'Latinoamerica', 'Jamaica': 'Latinoamerica', 'Dominican-Republic': 'Latinoamerica', 
    'Guatemala': 'Latinoamerica', 'Columbia': 'Latinoamerica', 'Haiti': 'Latinoamerica', 
    'Nicaragua': 'Latinoamerica', 'Peru': 'Latinoamerica', 'Ecuador': 'Latinoamerica', 
    'Trinadad&Tobago': 'Latinoamerica', 'Honduras': 'Latinoamerica', 
    'Argentina': 'Latinoamerica','Chile':'Latinoamerica','Brasil':'Latinoamerica','Venezuela':'Latinoamerica',
    
    # Europa
    'Germany': 'Europa', 'England': 'Europa', 'Italy': 'Europa', 'Poland': 'Europa', 
    'Portugal': 'Europa', 'Greece': 'Europa', 'France': 'Europa', 'Ireland': 'Europa', 
    'Yugoslavia': 'Europa', 'Scotland': 'Europa', 'Hungary': 'Europa', 'Holand-Netherlands': 'Europa',
    
    # Asia
    'Philippines': 'Asia', 'India': 'Asia', 'China': 'Asia', 'Japan': 'Asia', 
    'Vietnam': 'Asia', 'Taiwan': 'Asia', 'Iran': 'Asia', 'Hong Kong': 'Asia', 
    'Thailand': 'Asia', 'Cambodia': 'Asia', 'Laos': 'Asia', 'South-Korea': 'Asia',
    }

    X_copy['region'] = X_copy['native-country'].map(region_mapping)

    # (Opcional) Puedes eliminar la columna original si ya no la necesitas
    X_copy = X_copy.drop('native-country', axis=1)
    return X_copy

In [138]:
df_trat = regionalizar_paises(df_trat)
round(df_trat['region'].value_counts(normalize=True)*100,2)

region
EEUU                  91.40
Latinoamerica          4.32
Asia                   2.22
Europa                 1.63
Norteamerica_Otros     0.43
Name: proportion, dtype: float64

### Agrupación de nivel de educación

In [139]:
def agrupar_educacion(X):
    X_copy = X.copy()   
    education_mapping = {
        # Grupo 1: Educación Básica / Secundaria Incompleta
        'Preschool': 'Basica_Incompleta',
        '1st-4th':   'Basica_Incompleta',
        '5th-6th':   'Basica_Incompleta',
        '7th-8th':   'Basica_Incompleta',
        '9th':       'Basica_Incompleta',
        '10th':      'Basica_Incompleta',
        '11th':      'Basica_Incompleta',
        '12th':      'Basica_Incompleta',
        
        # Grupo 2: Secundaria Completa
        'HS-grad': 'Secundaria_Completa',
        
        # Grupo 3: Educación Técnica / Superior Incompleta
        'Some-college': 'Tecnica_o_Universitaria_Incompleta',
        'Assoc-voc':    'Tecnica_o_Universitaria_Incompleta',
        'Assoc-acdm':   'Tecnica_o_Universitaria_Incompleta',
        
        # Grupo 4: Grado Universitario
        'Bachelors': 'Universitaria',
        
        # Grupo 5: Posgrado y Especialidad
        'Masters':     'Maestría',
        'Prof-school': 'Escuela profesional',
        'Doctorate':   'Doctorado'
    }
    
    X_copy['nivel_educativo'] = X_copy['education'].map(education_mapping)
    
    columnas_a_eliminar = [col for col in ['education', 'education-num'] if col in X_copy.columns]
    X_copy = X_copy.drop(columnas_a_eliminar, axis=1)
    
    return X_copy

In [140]:
df_trat = agrupar_educacion(df_trat)
df_trat['nivel_educativo'].value_counts(normalize=True)*100

nivel_educativo
Secundaria_Completa                   32.316449
Tecnica_o_Universitaria_Incompleta    29.769461
Universitaria                         16.430531
Basica_Incompleta                     13.119856
Maestría                               5.439990
Escuela profesional                    1.707547
Doctorado                              1.216166
Name: proportion, dtype: float64

### Agrupación de Ocupaciones

Dentro de las ocupaciones, encontramos tres tipos de perfiles exactamente: 
- Perfil Administrativo, Técnico y Especializado
- Perfil Operativo y Manual
- Perfil de Servicios Generales
- Perfil Militar

Tras identificarlos, crearemos un agrupamiento permitiendo reducir las categorías y agrupando los perfiles similares.

In [141]:
def agrupar_ocupacion(X):
    X_copy = X.copy()

    occupation_mapping = {
        'Exec-managerial': 'Administrativo_Especializado',
        'Prof-specialty':  'Administrativo_Especializado',
        'Tech-support':    'Administrativo_Especializado',
        'Adm-clerical':    'Administrativo_Especializado',
        'Sales':           'Administrativo_Especializado',

        'Craft-repair':      'Operativo_Manual',
        'Machine-op-inspct': 'Operativo_Manual',
        'Transport-moving':  'Operativo_Manual',
        'Handlers-cleaners': 'Operativo_Manual',
        'Farming-fishing':   'Operativo_Manual',

        'Protective-serv': 'Servicios',
        'Other-service':   'Servicios',
        'Priv-house-serv': 'Servicios',

        'Armed-Forces': 'Otros_Militar',
    }
    
    # Mapeamos a la nueva columna
    X_copy['perfil_ocupacional'] = X_copy['occupation'].map(occupation_mapping)
       
    # Eliminamos la columna original para evitar multicolinealidad
    X_copy = X_copy.drop('occupation', axis=1)
    
    return X_copy

In [142]:
df_trat = agrupar_ocupacion(df_trat)
df_trat

,age,workclass,marital-status,relationship,race,sex,capital-gain,capital-loss,hours-per-week,income,income_num,region,nivel_educativo,perfil_ocupacional
0,39,State-gov,Never-married,Not-in-family,White,Male,2174,0,40,<=50K,0,EEUU,Universitaria,Administrativo_Especializado
1,50,Self-emp-not-inc,Married-civ-spouse,Husband,White,Male,0,0,13,<=50K,0,EEUU,Universitaria,Administrativo_Especializado
2,38,Private,Divorced,Not-in-family,White,Male,0,0,40,<=50K,0,EEUU,Secundaria_Completa,Operativo_Manual
3,53,Private,Married-civ-spouse,Husband,Black,Male,0,0,40,<=50K,0,EEUU,Basica_Incompleta,Operativo_Manual
4,28,Private,Married-civ-spouse,Wife,Black,Female,0,0,40,<=50K,0,Latinoamerica,Universitaria,Administrativo_Especializado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,Divorced,Not-in-family,White,Female,0,0,36,<=50K,0,EEUU,Universitaria,Administrativo_Especializado
48838,64,NaN,Widowed,Other-relative,Black,Male,0,0,40,<=50K,0,EEUU,Secundaria_Completa,NaN
48839,38,Private,Married-civ-spouse,Husband,White,Male,0,0,50,<=50K,0,EEUU,Universitaria,Administrativo_Especializado
48840,44,Private,Divorced,Own-child,Asian-Pac-Islander,Male,5455,0,40,<=50K,0,EEUU,Universitaria,Administrativo_Especializado


### Discretización de Capital-gain y Capital-loss usando Árboles de Decisión

Aplicando este método, reduciriamos los tiempos de ejecución. Por ello, es necesario aplicar árboles de decisión para obtener un corte más óptimo y discriminar objetivamente los rangos.

In [143]:
class DiscretizadorSupervisado(BaseEstimator, TransformerMixin):
    def __init__(self, max_depth=3, random_state=42):
        # Ya no pedimos las 'variables'. El ColumnTransformer se encargará de enviarlas.
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y=None):
        # 1. Validar que la variable objetivo (y) exista
        if y is None:
            raise ValueError("El DiscretizadorSupervisado requiere la variable objetivo 'y' durante el fit.")
            
        # 2. Convertir a DataFrame si el Pipeline nos pasa un Numpy Array
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        self.arboles_ = []
        
        # 3. Entrenar un árbol independiente por cada columna recibida
        for col in X.columns:
            arbol = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.random_state)
            arbol.fit(X[[col]], y)
            self.arboles_.append(arbol)
            
        return self

    def transform(self, X, y=None):
        # 1. Validar que el método fit ya fue ejecutado
        if not hasattr(self, "arboles_"):
            raise NotFittedError("Este transformador aún no ha sido entrenado (fit).")
            
        # 2. Asegurar formato DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
            
        X_copy = X.copy()
        
        # 3. Reemplazar los valores crudos por las probabilidades predecidas (la discretización)
        for i, col in enumerate(X.columns):
            # Extraemos la probabilidad de pertenecer a la clase 1 (>50K)
            X_copy[col] = self.arboles_[i].predict_proba(X[[col]])[:, 1]
            
        # Convertimos de vuelta a numpy array para evitar problemas con el ColumnTransformer
        return X_copy.to_numpy()
    
class DiscretizadorCapital(BaseEstimator, TransformerMixin):
    def __init__(self, max_depth=3, random_state=42):
        self.max_depth = max_depth
        self.random_state = random_state

    def fit(self, X, y=None):
        self._disc = DiscretizadorSupervisado(self.max_depth, self.random_state)
        self._disc.fit(X[['capital-gain', 'capital-loss']], y)

        # Aprender mapeos prob → etiqueta con datos de entrenamiento
        proba = self._disc.transform(X[['capital-gain', 'capital-loss']])
        proba_df = pd.DataFrame(proba, columns=['capital-gain', 'capital-loss'])
        self.mapeos_ = {
            col: {v: f"Nivel_{i+1}" for i, v in enumerate(sorted(proba_df[col].unique()))}
            for col in ['capital-gain', 'capital-loss']
        }
        return self

    def transform(self, X, y=None):
        X_copy = X.copy().reset_index(drop=True)
        proba = self._disc.transform(X_copy[['capital-gain', 'capital-loss']])
        proba_df = pd.DataFrame(proba, columns=['capital-gain', 'capital-loss'])
        X_copy['cat_capital-gain'] = proba_df['capital-gain'].map(self.mapeos_['capital-gain'])
        X_copy['cat_capital-loss'] = proba_df['capital-loss'].map(self.mapeos_['capital-loss'])
        return X_copy.drop(columns=['capital-gain', 'capital-loss'])

In [144]:
discretizador = DiscretizadorSupervisado(max_depth=3, random_state=42)

df_trat[['cat_capital-gain', 'cat_capital-loss']] = discretizador.fit_transform(
    df_trat[['capital-gain', 'capital-loss']], 
    df_trat['income_num'] # Pasamos la versión numérica
)

In [145]:
columnas_capital = ['cat_capital-gain', 'cat_capital-loss']
for col in columnas_capital:
    valores_unicos = sorted(df_trat[col].unique())

    mapeo_etiquetas = {valor: f"Nivel_{i+1}" for i, valor in enumerate(valores_unicos)}

    df_trat[col] = df_trat[col].map(mapeo_etiquetas)

df_trat = df_trat.drop('income_num', axis=1)

df_trat[['cat_capital-gain', 'cat_capital-loss', 'income']].head()

,cat_capital-gain,cat_capital-loss,income
0,Nivel_3,Nivel_4,<=50K
1,Nivel_3,Nivel_4,<=50K
2,Nivel_3,Nivel_4,<=50K
3,Nivel_3,Nivel_4,<=50K
4,Nivel_3,Nivel_4,<=50K


### Logarítmica + 1: Capital-gain y Capital-loss

Debido a la alta escala de las ganancias y pérdidas, utilizar una escala logarítmica es ideal; sin embargo, el dataset presenta muchos 0, lo cual es imposible calcular logarítmicamente dado que el $\ln(-x) = -\infty$  o bien $\lim_{x \to 0^+} \ln(-x) = -\infty$. Por ello, dado que no existen valores negativos, sumaremos una unidad a cada valor de manera que, si un valor era 0 inicialmente, este se volverá 1 y al aplicarle logaritmo, el valor final será 0 nuevamente. Además, permite minimizar la escala de los datos y acelerar el entrenamiento.



In [146]:
def log1p_capital(X):
    X_copy = X.copy()
    X_copy['log_capital_gain'] = np.log1p(X_copy['capital-gain'])
    X_copy['log_capital_loss'] = np.log1p(X_copy['capital-loss'])
    X_copy = X_copy.drop(columns = ['capital-gain','capital-loss'])
    return X_copy

In [147]:
df_trat = log1p_capital(df_trat)
df_trat

,age,workclass,marital-status,relationship,race,sex,hours-per-week,income,region,nivel_educativo,perfil_ocupacional,cat_capital-gain,cat_capital-loss,log_capital_gain,log_capital_loss
0,39,State-gov,Never-married,Not-in-family,White,Male,40,<=50K,EEUU,Universitaria,Administrativo_Especializado,Nivel_3,Nivel_4,7.684784,0.0
1,50,Self-emp-not-inc,Married-civ-spouse,Husband,White,Male,13,<=50K,EEUU,Universitaria,Administrativo_Especializado,Nivel_3,Nivel_4,0.000000,0.0
2,38,Private,Divorced,Not-in-family,White,Male,40,<=50K,EEUU,Secundaria_Completa,Operativo_Manual,Nivel_3,Nivel_4,0.000000,0.0
3,53,Private,Married-civ-spouse,Husband,Black,Male,40,<=50K,EEUU,Basica_Incompleta,Operativo_Manual,Nivel_3,Nivel_4,0.000000,0.0
4,28,Private,Married-civ-spouse,Wife,Black,Female,40,<=50K,Latinoamerica,Universitaria,Administrativo_Especializado,Nivel_3,Nivel_4,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,Divorced,Not-in-family,White,Female,36,<=50K,EEUU,Universitaria,Administrativo_Especializado,Nivel_3,Nivel_4,0.000000,0.0
48838,64,NaN,Widowed,Other-relative,Black,Male,40,<=50K,EEUU,Secundaria_Completa,NaN,Nivel_3,Nivel_4,0.000000,0.0
48839,38,Private,Married-civ-spouse,Husband,White,Male,50,<=50K,EEUU,Universitaria,Administrativo_Especializado,Nivel_3,Nivel_4,0.000000,0.0
48840,44,Private,Divorced,Own-child,Asian-Pac-Islander,Male,40,<=50K,EEUU,Universitaria,Administrativo_Especializado,Nivel_4,Nivel_4,8.604471,0.0


## Partición de datos: Train (80%) y Test (20%)

In [148]:
# 1. Separar features y target
X_raw = df_raw.drop(columns=['income'])
y_raw = df_raw['income'].map({'<=50K': 0, '>50K': 1})

# 2. Partir ANTES de tocar los pipelines
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

## Pipeline

In [149]:
def imputar_desconocido(X):
    X_copy = X.copy()
    for col in ['workclass', 'region', 'perfil_ocupacional']:
        if col in X_copy.columns:
            X_copy[col] = X_copy[col].fillna('Desconocido')
    return X_copy


_pasos_comunes = [
    ('regionalizar',      FunctionTransformer(regionalizar_paises)),
    ('agrupar_educacion', FunctionTransformer(agrupar_educacion)),
    ('agrupar_ocupacion', FunctionTransformer(agrupar_ocupacion)),
    ('imputar_desconocido', FunctionTransformer(imputar_desconocido))
]

_cat_base = ['workclass', 'marital-status', 'relationship', 'race', 'sex',
             'region', 'nivel_educativo', 'perfil_ocupacional']

_col_disc = {'num': ['age', 'hours-per-week'],
             'cat': _cat_base + ['cat_capital-gain', 'cat_capital-loss']}

_col_log  = {'num': ['age', 'hours-per-week', 'log_capital_gain', 'log_capital_loss'],
             'cat': _cat_base}


def _build_ct(cols):
    return ColumnTransformer([
        ('num', StandardScaler(),                                              cols['num']),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),   cols['cat'])
    ], remainder='drop')


# ── Pasos compartidos ─────────────────────────────────────────────────────────
_pasos_comunes = [
    ('regionalizar',      FunctionTransformer(regionalizar_paises)),
    ('agrupar_educacion', FunctionTransformer(agrupar_educacion)),
    ('agrupar_ocupacion', FunctionTransformer(agrupar_ocupacion)),
    ('imputar',           FunctionTransformer(imputar_desconocido)),
]

# ── Pipelines ─────────────────────────────────────────────────────────────────
pipeline_disc = Pipeline(_pasos_comunes + [
    ('discretizar_capital', DiscretizadorCapital()),
    ('encoding_scaling',    _build_ct(_col_disc)),
])

pipeline_log = Pipeline(_pasos_comunes + [
    ('log1p_capital',    FunctionTransformer(log1p_capital)),
    ('encoding_scaling', _build_ct(_col_log)),
])


In [160]:
X_train_disc = pipeline_disc.fit_transform(X_train, y_train)
X_test_disc  = pipeline_disc.transform(X_test)

X_train_log  = pipeline_log.fit_transform(X_train, y_train)
X_test_log   = pipeline_log.transform(X_test)

In [156]:
pd.DataFrame(X_train_disc)

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,-0.120090,-0.844835,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1.268245,-0.038454,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.049034,-0.441645,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,2.437369,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.537542,-0.199731,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39068,0.464472,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
39069,1.414385,-1.651216,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
39070,1.633596,-0.038454,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
39071,-0.047020,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [162]:
pd.DataFrame(X_test_disc)

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,-0.631582,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1.122104,-0.119092,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-1.289214,-0.441645,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.266231,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.245261,0.364736,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9764,-0.120090,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9765,-0.339301,0.767926,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9766,1.487456,-0.844835,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9767,-0.704652,-0.038454,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
